# 002 OpenAI Chat History

这是第二份 OpenAI 学习 Notebook。

学习目标：

1. 理解“多轮对话”为什么需要上下文
2. 学会用 `messages` 保存历史对话
3. 学会用 Chat Completions API 实现私有网关兼容的多轮聊天
4. 对比 OpenAI 官方 Responses API 的 `previous_response_id` 思路

你当前使用的是私有模型网关：`OPENAI_BASE_URL=http://192.168.102.19:8082/v1`，模型是 `qwq`。

这个网关已验证支持 `/v1/chat/completions`，所以本 Notebook 主要使用 `messages` 历史数组来维护上下文。


## 先理解核心概念

单轮聊天只需要一条用户问题：

```python
"你好，请介绍 FastAPI"
```

多轮聊天需要把之前的内容一起传给模型，例如：

```python
[
    {"role": "system", "content": "你是教学助手"},
    {"role": "user", "content": "我正在学 FastAPI"},
    {"role": "assistant", "content": "FastAPI 是一个 Python Web 框架"},
    {"role": "user", "content": "它和 Spring Boot 有什么区别？"},
]
```

模型本身不会天然知道上一轮对话。你必须通过上下文告诉它。

在 Java 项目里，可以把 `messages` 理解成一次请求传给模型的“会话上下文 DTO”。


## 加载环境变量

这里和第一份 Notebook 保持一致：自动向上查找项目根目录 `.env`。


In [39]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)


Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 创建客户端

私有兼容网关一般只要求：

- `api_key` 非空
- `base_url` 指向网关地址
- `model` 使用网关里存在的模型名

你当前模型是 `qwq`。


In [40]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


SYSTEM_PROMPT = """
你是一个教学型聊天助手。
用户是 Java 开发者，正在学习 Python、FastAPI 和 AI 应用开发。
回答时请使用中文，尽量结合 Java 类比，但不要太啰嗦。
""".strip()


## 初始化会话历史

`conversation_messages` 就是当前 Notebook 里的“会话状态”。

它是一个 list，每个元素都是一条消息。

消息里的 `role` 常见有：

- `system`：系统指令，告诉模型应该扮演什么角色
- `user`：用户输入
- `assistant`：模型回复

后续每次提问，我们都会把用户问题和模型回答追加到这个列表里。


In [41]:
conversation_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
]

conversation_messages


[{'role': 'system',
  'content': '你是一个教学型聊天助手。\n用户是 Java 开发者，正在学习 Python、FastAPI 和 AI 应用开发。\n回答时请使用中文，尽量结合 Java 类比，但不要太啰嗦。'}]

## 封装多轮聊天函数

这个函数做 4 件事：

1. 把用户问题追加到 `conversation_messages`
2. 把完整历史发给模型
3. 取出模型回复
4. 把模型回复也追加到 `conversation_messages`

这样下一轮提问时，模型就能看到前面的上下文。


In [42]:
def chat_with_history(message: str) -> str:
    conversation_messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=conversation_messages,
    )

    reply = response.choices[0].message.content or ""
    conversation_messages.append({"role": "assistant", "content": reply})
    return reply


## 第一轮对话

先告诉模型你的学习背景。


In [43]:
reply = chat_with_history("我是 Java 开发者，正在学习 FastAPI。请用 3 句话介绍 FastAPI。")
print(reply)




FastAPI 是一个基于 Python 类型注解的高性能异步 Web 框架，性能可对标 Spring Boot，但底层更轻量、启动更快。它像 Spring Boot 集成 Swagger 一样开箱即用，只需在路由函数上标注参数类型，即可自动生成 OpenAPI 文档与交互式调试界面，彻底告别手写接口说明。作为 Python 原生框架，它能无缝调用 PyTorch、NumPy 等 AI 库，非常适合你从 Java 生态迁移后快速构建高性能 AI 推理服务或微服务。


## 第二轮对话

这次问题里没有再次说明“我是 Java 开发者”。

如果上下文生效，模型应该仍然知道你的背景。


In [44]:
reply = chat_with_history("那它和 Spring Boot 最核心的区别是什么？")
print(reply)




最核心的区别在于**并发模型与生态定位**：FastAPI 原生基于 Python 的 `async/await`，专为高并发 I/O 和 AI 场景设计，靠类型注解自动完成数据校验，代码极简；Spring Boot 基于 JVM 线程池模型（同步阻塞为主），是为企业级复杂业务、事务管理和完整生态量身定制的“重型框架”。类比来说，FastAPI 像 Spring Boot 的“轻量极客版”，省去了大量注解与配置，但弱化了传统企业级功能（如 Spring Security 的完整体系、分布式事务等），更适合快速构建 AI 接口或微服务。


## 查看当前会话历史

这里可以看到多轮对话是如何被保存的。

真实项目里，这些数据通常会落到数据库，比如：

- `chat_sessions`
- `chat_messages`


In [45]:
from pprint import pprint

pprint(conversation_messages)


[{'content': '你是一个教学型聊天助手。\n'
             '用户是 Java 开发者，正在学习 Python、FastAPI 和 AI 应用开发。\n'
             '回答时请使用中文，尽量结合 Java 类比，但不要太啰嗦。',
  'role': 'system'},
 {'content': '我是 Java 开发者，正在学习 FastAPI。请用 3 句话介绍 FastAPI。', 'role': 'user'},
 {'content': '\n'
             '\n'
             'FastAPI 是一个基于 Python 类型注解的高性能异步 Web 框架，性能可对标 Spring '
             'Boot，但底层更轻量、启动更快。它像 Spring Boot 集成 Swagger '
             '一样开箱即用，只需在路由函数上标注参数类型，即可自动生成 OpenAPI 文档与交互式调试界面，彻底告别手写接口说明。作为 '
             'Python 原生框架，它能无缝调用 PyTorch、NumPy 等 AI 库，非常适合你从 Java 生态迁移后快速构建高性能 '
             'AI 推理服务或微服务。',
  'role': 'assistant'},
 {'content': '那它和 Spring Boot 最核心的区别是什么？', 'role': 'user'},
 {'content': '\n'
             '\n'
             '最核心的区别在于**并发模型与生态定位**：FastAPI 原生基于 Python 的 `async/await`，专为高并发 '
             'I/O 和 AI 场景设计，靠类型注解自动完成数据校验，代码极简；Spring Boot 基于 JVM '
             '线程池模型（同步阻塞为主），是为企业级复杂业务、事务管理和完整生态量身定制的“重型框架”。类比来说，FastAPI 像 '
             'Spring Boot 的“轻量极客版”，省去了大量注解与配置，但弱化了传统企业级功能（如 Spring Sec

## 只看最近几轮历史

随着对话变长，`messages` 会越来越大。

真实项目里不应该无限制把所有历史都传给模型，否则会遇到：

1. token 越来越多
2. 请求越来越慢
3. 成本越来越高
4. 超过模型上下文长度

一个简单做法是：只保留 system prompt 和最近 N 轮对话。


In [46]:
def build_recent_messages(messages: list[dict], recent_turns: int = 3) -> list[dict]:
    system_messages = [item for item in messages if item["role"] == "system"]
    non_system_messages = [item for item in messages if item["role"] != "system"]
    recent_messages = non_system_messages[-recent_turns * 2:]
    return system_messages + recent_messages


recent_messages = build_recent_messages(conversation_messages, recent_turns=2)
pprint(recent_messages)


[{'content': '你是一个教学型聊天助手。\n'
             '用户是 Java 开发者，正在学习 Python、FastAPI 和 AI 应用开发。\n'
             '回答时请使用中文，尽量结合 Java 类比，但不要太啰嗦。',
  'role': 'system'},
 {'content': '我是 Java 开发者，正在学习 FastAPI。请用 3 句话介绍 FastAPI。', 'role': 'user'},
 {'content': '\n'
             '\n'
             'FastAPI 是一个基于 Python 类型注解的高性能异步 Web 框架，性能可对标 Spring '
             'Boot，但底层更轻量、启动更快。它像 Spring Boot 集成 Swagger '
             '一样开箱即用，只需在路由函数上标注参数类型，即可自动生成 OpenAPI 文档与交互式调试界面，彻底告别手写接口说明。作为 '
             'Python 原生框架，它能无缝调用 PyTorch、NumPy 等 AI 库，非常适合你从 Java 生态迁移后快速构建高性能 '
             'AI 推理服务或微服务。',
  'role': 'assistant'},
 {'content': '那它和 Spring Boot 最核心的区别是什么？', 'role': 'user'},
 {'content': '\n'
             '\n'
             '最核心的区别在于**并发模型与生态定位**：FastAPI 原生基于 Python 的 `async/await`，专为高并发 '
             'I/O 和 AI 场景设计，靠类型注解自动完成数据校验，代码极简；Spring Boot 基于 JVM '
             '线程池模型（同步阻塞为主），是为企业级复杂业务、事务管理和完整生态量身定制的“重型框架”。类比来说，FastAPI 像 '
             'Spring Boot 的“轻量极客版”，省去了大量注解与配置，但弱化了传统企业级功能（如 Spring Sec

## 使用最近历史继续对话

这个函数不会把全部历史传给模型，只传最近几轮。

这是很多业务系统早期版本会采用的简单策略。


In [ ]:
def chat_with_recent_history(message: str, recent_turns: int = 3) -> str:
    conversation_messages.append({"role": "user", "content": message})
    request_messages = build_recent_messages(conversation_messages, recent_turns=recent_turns)

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=request_messages,
    )

    reply = response.choices[0].message.content or ""
    conversation_messages.append({"role": "assistant", "content": reply})
    return reply


In [ ]:
reply = chat_with_recent_history("请给我一个适合初学者的 FastAPI 学习顺序。", recent_turns=2)
print(reply)


## 清空会话

Notebook 学习时，经常需要重新开始一轮对话。

这时可以把历史重置回只有 `system` 消息。


In [ ]:
def reset_conversation() -> None:
    conversation_messages.clear()
    conversation_messages.append({"role": "system", "content": SYSTEM_PROMPT})


reset_conversation()
conversation_messages


## OpenAI 官方 Responses API 的另一种思路

如果使用 OpenAI 官方 `Responses API`，可以学习 `previous_response_id`。

大致思路是：

1. 第一轮请求返回一个 `response.id`
2. 第二轮请求带上 `previous_response_id`
3. OpenAI 官方服务端会帮你关联前后文

伪代码示例：

```python
first = client.responses.create(
    model=OPENAI_MODEL,
    input="我是 Java 开发者，正在学习 FastAPI"
)

second = client.responses.create(
    model=OPENAI_MODEL,
    previous_response_id=first.id,
    input="那它和 Spring Boot 有什么区别？"
)
```

但是你的当前私有网关不支持 `/v1/responses`，所以这份 Notebook 采用 `messages` 历史方式。


## 当前阶段结论

你现在需要记住：

1. 多轮对话的本质是“把上下文传给模型”
2. 私有 OpenAI 兼容网关通常优先使用 `messages` 历史
3. 历史不能无限增长，真实项目要做截断、摘要或持久化
4. 后续接入 FastAPI 时，`conversation_messages` 不应该只放在内存里，而应该设计表存储

下一份建议学习：

- `003-openai-structured-output.ipynb`

主题是：让模型稳定输出 JSON，为股票智能体做意图识别。
